# KataGo v1.17 Contribution on Modal

This notebook deploys the dedicated Modal service used by SWHub's **KataGo Contribution** page in normal release builds. It is separate from `Kata_Modal.ipynb`, which is only for interactive analysis.

The contributor runs KataGo's official `contribute` command on one Modal T4, receives the current model and self-play tasks from katagotraining.org, and streams game snapshots back to SWHub over a WebSocket.

Before starting:

1. Create an account at https://katagotraining.org/accounts/signup/.
2. Use a unique password for that account. SWHub saves the training account in local shared preferences and sends it when a session starts.
3. Keep the generated endpoint private. By default anyone with its URL can cold-start the paid container; the optional endpoint-login step below can require a separate username and password.
4. Remember that Modal GPU time is metered. Use **Finish games & stop** in SWHub, and stop the Modal app when finished.


## 1. Install Modal


In [ ]:
%uv pip install -q --upgrade modal

## 2. Log in to Modal

Run this cell and follow the printed browser/device-code flow.


In [ ]:
import os, re, subprocess

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"

login = subprocess.Popen(
    ["modal", "token", "new"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
for line in login.stdout:
    print(line, end="")
    match = re.search(r"https://modal\.com/token-flow/[A-Za-z0-9_-]+", line)
    if match:
        print("\nOPEN THIS LINK NOW:\n" + match.group(0))
if login.wait() != 0:
    raise RuntimeError("Modal login failed")


## 3. Write the Modal app

This notebook is self-contained. Running the next cell writes the complete audited KataGo v1.17 contribution service into the notebook runtime; no repository checkout or separate Python download is required.


In [ ]:
%%writefile modal_katago_contribute.py
"""KataGo v1.17 distributed-training contributor for SWHub on Modal.

Generated and deployed by Kata_Modal_Contribution.ipynb. To redeploy after
running this cell, use: modal deploy modal_katago_contribute.py

Then paste the printed wss:// URL plus /contribute into SWHub's
KataGo Contribution page.
"""

import asyncio
import hashlib
import hmac
import json
import os
import re
import shutil
import subprocess
import tempfile
import urllib.request
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import modal


APP_NAME = "Katago-Contribute"
MODAL_GPU = "T4"
SESSION_TIMEOUT_HOURS = 4

KATAGO_RELEASE = "v1.17.1"
KATAGO_URL = (
    "https://github.com/lightvector/KataGo/releases/download/v1.17.1/"
    "katago-v1.17.1-cuda12.8-cudnn9.8.0-linux-x64.zip"
)
KATAGO_SHA256 = "458d226c2c8533600251bba3b2ee612d3aee0c796f592a2b53839a6a05b0826e"

CACHE = Path("/cache")
KATAGO_DIR = CACHE / "katago_v1_17_1_cuda12_8_cudnn9"
KATAGO_ARCHIVE = KATAGO_DIR / "katago.zip"
KATAGO_BIN = KATAGO_DIR / "katago"
KATAGO_EXTRACTED_BIN = KATAGO_DIR / "squashfs-root" / "AppRun"
CONTRIBUTE_BASE_DIR = CACHE / "contribute"
HOME_DATA_DIR = CACHE / "home"
LOG_DIR = CACHE / "logs"

app = modal.App(APP_NAME)
volume = modal.Volume.from_name("katago-contribute", create_if_missing=True)
if modal.is_local():
    endpoint_auth_secret = modal.Secret.from_dict(
        {
            "SWHUB_CONTRIBUTOR_USERNAME": os.environ.get(
                "SWHUB_CONTRIBUTOR_USERNAME", ""
            ),
            "SWHUB_CONTRIBUTOR_PASSWORD": os.environ.get(
                "SWHUB_CONTRIBUTOR_PASSWORD", ""
            ),
        }
    )
else:
    endpoint_auth_secret = modal.Secret.from_dict({})
image = (
    modal.Image.from_registry(
        "nvidia/cuda:12.8.1-cudnn-runtime-ubuntu22.04",
        add_python="3.11",
    )
    .entrypoint([])
    .apt_install(
        "ca-certificates",
        "curl",
        "libgomp1",
        "libomp-dev",
        "libzip4",
        "p7zip-full",
        "unzip",
        "wget",
    )
    .pip_install("fastapi[standard]")
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _download(url: str, target: Path, expected_sha256: str) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        if _sha256(target) == expected_sha256:
            return
        target.unlink()
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.unlink(missing_ok=True)
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "SWHub-KataGo-Modal-Contributor/1.0",
            "Accept": "*/*",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with temporary.open("wb") as output:
                shutil.copyfileobj(response, output)
    except Exception:
        temporary.unlink(missing_ok=True)
        subprocess.run(
            [
                "wget",
                "--tries=3",
                "--timeout=60",
                "--user-agent=SWHub-KataGo-Modal-Contributor/1.0",
                "-O",
                str(temporary),
                url,
            ],
            check=True,
        )
    if not temporary.exists() or temporary.stat().st_size == 0:
        raise RuntimeError(f"Download produced an empty file: {url}")
    actual_sha256 = _sha256(temporary)
    if actual_sha256 != expected_sha256:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Download checksum mismatch for {url}: {actual_sha256}"
        )
    temporary.replace(target)


def _command_output(
    arguments: list[str],
    cwd: Path | None = None,
    timeout: int = 120,
) -> tuple[int, str]:
    result = subprocess.run(
        arguments,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
        timeout=timeout,
    )
    return result.returncode, result.stdout


def ensure_katago() -> Path:
    KATAGO_DIR.mkdir(parents=True, exist_ok=True)
    if not KATAGO_BIN.exists():
        _download(KATAGO_URL, KATAGO_ARCHIVE, KATAGO_SHA256)
        with zipfile.ZipFile(KATAGO_ARCHIVE) as archive:
            archive.extractall(KATAGO_DIR)
        KATAGO_ARCHIVE.unlink(missing_ok=True)
    if not KATAGO_BIN.exists():
        raise RuntimeError(f"KataGo archive did not contain {KATAGO_BIN}")
    KATAGO_BIN.chmod(0o755)

    code, output = _command_output([str(KATAGO_BIN), "version"])
    if code == 0 and KATAGO_RELEASE.removeprefix("v") in output:
        return KATAGO_BIN

    if not KATAGO_EXTRACTED_BIN.exists():
        code, extract_output = _command_output(
            [str(KATAGO_BIN), "--appimage-extract"],
            cwd=KATAGO_DIR,
        )
        if code != 0:
            raise RuntimeError(
                "KataGo AppImage extraction failed:\n" + extract_output
            )
    KATAGO_EXTRACTED_BIN.chmod(0o755)
    code, output = _command_output([str(KATAGO_EXTRACTED_BIN), "version"])
    if code != 0 or KATAGO_RELEASE.removeprefix("v") not in output:
        raise RuntimeError("KataGo v1.17 validation failed:\n" + output)
    return KATAGO_EXTRACTED_BIN


def _bool(value: object) -> str:
    return "true" if value is True else "false"


def _quoted(value: object) -> str:
    return json.dumps(str(value), ensure_ascii=False)


def _int_in_range(
    config: dict,
    key: str,
    default: int,
    minimum: int,
    maximum: int,
) -> int:
    try:
        value = int(config.get(key, default))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{key} must be a number") from exc
    if value < minimum or value > maximum:
        raise ValueError(f"{key} must be between {minimum} and {maximum}")
    return value


def _tri_state(config: dict, key: str) -> str:
    value = str(config.get(key, "auto")).lower()
    if value not in {"auto", "true", "false"}:
        raise ValueError(f"{key} must be auto, true, or false")
    return value


_PROTECTED_EXTRA_KEYS = {
    "cudadevicetouse",
    "cudausefp16",
    "cudausenhwc",
    "disablepredownloadloop",
    "homedatadir",
    "includeownership",
    "logdir",
    "loggamesasjson",
    "maxratingmatches",
    "maxsimultaneousgames",
    "mirroruseproxy",
    "modeldownloadmirrorbaseurl",
    "numnnserverthreadspermodel",
    "onlyplayratingmatches",
    "password",
    "proxybasicauthpassword",
    "proxybasicauthusername",
    "proxyhost",
    "proxyport",
    "reportperformanceevery",
    "serverurl",
    "taskrepfactor",
    "username",
    "warntaskunusedkeys",
    "watchongoinggameinfile",
    "watchongoinggameinfilename",
}


def _safe_extra_lines(raw: object) -> list[str]:
    raw_text = str(raw or "")
    if len(raw_text) > 16000:
        raise ValueError("Additional config must be 16,000 characters or fewer")
    lines: list[str] = []
    for line in raw_text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            lines.append(line)
            continue
        match = re.match(r"^([A-Za-z][A-Za-z0-9_]*)\s*=", stripped)
        if not match:
            raise ValueError(
                f"Invalid additional config line (expected key = value): {line}"
            )
        if match.group(1).lower() in _PROTECTED_EXTRA_KEYS:
            raise ValueError(
                f"{match.group(1)} is managed by SWHub and cannot be overridden"
            )
        lines.append(line)
    return lines


def build_config(config: dict) -> str:
    server_url = str(config.get("serverUrl", "")).strip()
    parsed_server_url = urlparse(server_url)
    if (
        parsed_server_url.scheme != "https"
        or not parsed_server_url.netloc
        or parsed_server_url.username is not None
        or parsed_server_url.password is not None
    ):
        raise ValueError("serverUrl must be a valid https:// URL")
    username = str(config.get("username", "")).strip()
    password = str(config.get("password", ""))
    if not username or not password:
        raise ValueError("username and password are required")

    proxy_host = str(config.get("proxyHost", "")).strip()
    proxy_username = str(config.get("proxyBasicAuthUsername", "")).strip()
    proxy_password = str(config.get("proxyBasicAuthPassword", ""))
    mirror_url = str(config.get("modelDownloadMirrorBaseUrl", "")).strip()
    if mirror_url:
        parsed_mirror_url = urlparse(mirror_url)
        if (
            parsed_mirror_url.scheme != "https"
            or not parsed_mirror_url.netloc
            or parsed_mirror_url.username is not None
            or parsed_mirror_url.password is not None
        ):
            raise ValueError("modelDownloadMirrorBaseUrl must be https://")
    watch_name = Path(
        str(config.get("watchOngoingGameInFileName") or "watchgame.txt")
    ).name
    max_games = _int_in_range(
        config, "maxSimultaneousGames", 8, 1, 128
    )
    max_rating = _int_in_range(config, "maxRatingMatches", 1, 0, 128)
    task_rep_factor = _int_in_range(config, "taskRepFactor", 4, 2, 16)
    report_performance_every = _int_in_range(
        config, "reportPerformanceEvery", 120, 1, 21600
    )
    nn_threads = _int_in_range(
        config, "numNNServerThreadsPerModel", 1, 1, 16
    )
    cuda_device = _int_in_range(config, "cudaDeviceToUse", 0, 0, 15)

    values = [
        f"serverUrl = {_quoted(server_url)}",
        f"username = {_quoted(username)}",
        f"password = {_quoted(password)}",
        f"maxSimultaneousGames = {max_games}",
        "logGamesAsJson = true",
        f"includeOwnership = {_bool(config.get('includeOwnership', False))}",
        "watchOngoingGameInFile = "
        + _bool(config.get("watchOngoingGameInFile", False)),
        f"watchOngoingGameInFileName = {_quoted(watch_name)}",
        "onlyPlayRatingMatches = "
        + _bool(config.get("onlyPlayRatingMatches", False)),
        f"maxRatingMatches = {max_rating}",
        "disablePredownloadLoop = "
        + _bool(config.get("disablePredownloadLoop", False)),
        f"taskRepFactor = {task_rep_factor}",
        f"reportPerformanceEvery = {report_performance_every}",
        "warnTaskUnusedKeys = "
        + _bool(config.get("warnTaskUnusedKeys", False)),
        f"numNNServerThreadsPerModel = {nn_threads}",
        f"cudaDeviceToUse = {cuda_device}",
        f"cudaUseFP16 = {_tri_state(config, 'cudaUseFP16')}",
        f"cudaUseNHWC = {_tri_state(config, 'cudaUseNHWC')}",
        f"homeDataDir = {_quoted(HOME_DATA_DIR)}",
        f"logDir = {_quoted(LOG_DIR)}",
    ]
    if proxy_host:
        proxy_port = _int_in_range(config, "proxyPort", 0, 1, 65535)
        values.extend(
            [
                f"proxyHost = {_quoted(proxy_host)}",
                f"proxyPort = {proxy_port}",
            ]
        )
        if proxy_username:
            values.extend(
                [
                    f"proxyBasicAuthUsername = {_quoted(proxy_username)}",
                    f"proxyBasicAuthPassword = {_quoted(proxy_password)}",
                ]
            )
    if mirror_url:
        values.extend(
            [
                f"modelDownloadMirrorBaseUrl = {_quoted(mirror_url)}",
                "mirrorUseProxy = "
                + _bool(config.get("mirrorUseProxy", True)),
            ]
        )
    values.extend(_safe_extra_lines(config.get("extraConfig")))
    return "\n".join(values) + "\n"


@app.function(
    image=image,
    gpu=MODAL_GPU,
    cpu=4,
    memory=8192,
    timeout=SESSION_TIMEOUT_HOURS * 60 * 60,
    startup_timeout=20 * 60,
    max_containers=1,
    scaledown_window=60,
    volumes={"/cache": volume},
    secrets=[endpoint_auth_secret],
)
@modal.concurrent(max_inputs=1)
@modal.asgi_app()
def gg():
    from fastapi import FastAPI, WebSocket, WebSocketDisconnect

    web_app = FastAPI()

    @web_app.get("/")
    async def status():
        return {
            "ok": True,
            "app": APP_NAME,
            "katago": KATAGO_RELEASE,
            "gpu": MODAL_GPU,
            "protocol": "swhub-katago-contribution",
            "protocolVersion": 1,
            "websocket": "/contribute",
            "note": "Connect with the SWHub KataGo Contribution page.",
        }

    @web_app.websocket("/contribute")
    async def contribute(websocket: WebSocket):
        await websocket.accept()
        process: asyncio.subprocess.Process | None = None
        config_path: Path | None = None
        reader_tasks: list[asyncio.Task] = []
        uploaded_games = 0
        latest_games: dict[str, dict] = {}
        send_lock = asyncio.Lock()
        shutting_down = False
        paused = False
        sensitive_values: list[str] = []

        def redact(text: str) -> str:
            result = text
            for value in sensitive_values:
                if value:
                    result = result.replace(value, "[REDACTED]")
            return result

        async def send(event_type: str, **data) -> None:
            payload = {"type": event_type, **data}
            async with send_lock:
                await websocket.send_text(
                    json.dumps(payload, separators=(",", ":"))
                )

        async def write_command(command: str) -> None:
            if process is None or process.stdin is None:
                return
            process.stdin.write((command + "\n").encode())
            await process.stdin.drain()

        async def commit_cache() -> None:
            try:
                await asyncio.to_thread(volume.commit)
            except Exception as exc:
                message = f"Volume commit warning: {exc!r}"
                print(message, flush=True)
                try:
                    await send("log", message=message)
                except Exception:
                    pass

        async def periodic_commit() -> None:
            while process is not None and process.returncode is None:
                await asyncio.sleep(10 * 60)
                await commit_cache()

        def finish_data_from_upload_log(
            log_line: str,
        ) -> tuple[str, dict] | None:
            match = re.search(
                r"\buploaded sgf\s+(\S+\.sgf)(?:\s|$)",
                log_line,
                flags=re.IGNORECASE,
            )
            if match is None:
                return None
            sgf_path = Path(match.group(1))
            finish_data: dict[str, object] = {"finished": True}
            try:
                sgf_text = sgf_path.read_text(
                    encoding="utf-8", errors="replace"
                )
            except OSError:
                return sgf_path.stem, finish_data
            result_match = re.search(r"RE\[([^\]]*)\]", sgf_text)
            if result_match is None:
                return sgf_path.stem, finish_data
            result = result_match.group(1).strip()
            if result:
                finish_data["result"] = result
                upper_result = result.upper()
                if upper_result.startswith("B+"):
                    finish_data["winner"] = "B"
                elif upper_result.startswith("W+"):
                    finish_data["winner"] = "W"
                finish_data["resigned"] = upper_result.endswith("+R")
            return sgf_path.stem, finish_data

        async def read_stdout() -> None:
            nonlocal uploaded_games
            assert process is not None and process.stdout is not None
            while True:
                line = await process.stdout.readline()
                if not line:
                    return
                raw_text = line.decode(errors="ignore").strip()
                text = redact(raw_text)
                if not text:
                    continue
                if text.startswith("{"):
                    try:
                        data = json.loads(text)
                        if isinstance(data, dict) and data.get("gameId"):
                            latest_games[str(data["gameId"])] = dict(data)
                            await send("game", data=data)
                            continue
                    except json.JSONDecodeError:
                        pass
                if "uploaded sgf" in text.lower():
                    finish = await asyncio.to_thread(
                        finish_data_from_upload_log, raw_text
                    )
                    if finish is not None:
                        game_id, finish_data = finish
                        latest = latest_games.pop(game_id, None)
                        if latest is not None:
                            finished = {**latest, **finish_data}
                            await send("game", data=finished)
                    uploaded_games += 1
                    await send(
                        "uploaded",
                        uploadedGames=uploaded_games,
                        message=f"Uploaded game {uploaded_games}",
                    )
                else:
                    await send("log", stream="stdout", message=text)

        async def read_stderr() -> None:
            assert process is not None and process.stderr is not None
            while True:
                line = await process.stderr.readline()
                if not line:
                    return
                text = redact(line.decode(errors="ignore").strip())
                if not text:
                    continue
                lower = text.lower()
                event_type = (
                    "error"
                    if "uncaught exception" in lower
                    or "server returned error" in lower
                    or "not status code 200 ok" in lower
                    else "log"
                )
                await send(event_type, stream="stderr", message=text)

        async def drain_output_readers() -> None:
            output_readers = reader_tasks[:2]
            if not output_readers:
                return
            _, pending = await asyncio.wait(output_readers, timeout=10)
            for task in pending:
                task.cancel()
            await asyncio.gather(*output_readers, return_exceptions=True)

        async def stop_process(force: bool) -> None:
            nonlocal shutting_down, paused
            if process is None or process.returncode is not None:
                return
            shutting_down = True
            try:
                if not force and paused:
                    await write_command("resume")
                    paused = False
                await write_command("forcequit" if force else "quit")
            except (BrokenPipeError, ConnectionResetError):
                pass
            try:
                await asyncio.wait_for(
                    process.wait(),
                    timeout=15 if force else 120,
                )
            except asyncio.TimeoutError:
                try:
                    process.terminate()
                except ProcessLookupError:
                    return
                try:
                    await asyncio.wait_for(process.wait(), timeout=10)
                except asyncio.TimeoutError:
                    try:
                        process.kill()
                    except ProcessLookupError:
                        return
                    await process.wait()

        try:
            await send(
                "hello",
                protocol="swhub-katago-contribution",
                protocolVersion=1,
                message=f"Modal {MODAL_GPU} contributor ready for KataGo {KATAGO_RELEASE}.",
            )
            raw = await asyncio.wait_for(websocket.receive_text(), timeout=60)
            try:
                request = json.loads(raw)
            except json.JSONDecodeError:
                await send("error", message="Start request must be valid JSON.")
                await websocket.close(code=4400)
                return
            if not isinstance(request, dict):
                await send("error", message="Start request must be a JSON object.")
                await websocket.close(code=4400)
                return
            if request.get("protocolVersion") != 1:
                await send(
                    "error",
                    message="Unsupported SWHub contribution protocol version.",
                )
                await websocket.close(code=4400)
                return
            expected_endpoint_username = os.environ.get(
                "SWHUB_CONTRIBUTOR_USERNAME", ""
            )
            expected_endpoint_password = os.environ.get(
                "SWHUB_CONTRIBUTOR_PASSWORD", ""
            )
            endpoint_auth = request.get("endpointAuth")
            if bool(expected_endpoint_username) != bool(expected_endpoint_password):
                await send(
                    "error",
                    message="Modal endpoint login is configured incompletely.",
                )
                await websocket.close(code=1011)
                return
            if expected_endpoint_username:
                endpoint_username = (
                    str(endpoint_auth.get("username", ""))
                    if isinstance(endpoint_auth, dict)
                    else ""
                )
                endpoint_password = (
                    str(endpoint_auth.get("password", ""))
                    if isinstance(endpoint_auth, dict)
                    else ""
                )
                if not (
                    hmac.compare_digest(
                        endpoint_username, expected_endpoint_username
                    )
                    and hmac.compare_digest(
                        endpoint_password, expected_endpoint_password
                    )
                ):
                    await send("error", message="Contributor login failed.")
                    await websocket.close(code=4401)
                    return
            if request.get("action") != "start" or not isinstance(
                request.get("config"), dict
            ):
                await send("error", message="First message must be a start request.")
                await websocket.close(code=4400)
                return
            sensitive_values.extend(
                [
                    str(request["config"].get("password", "")),
                    str(request["config"].get("proxyBasicAuthPassword", "")),
                    str(endpoint_auth.get("password", ""))
                    if isinstance(endpoint_auth, dict)
                    else "",
                ]
            )

            await send("connecting", message="Preparing KataGo v1.17 in Modal…")
            katago = await asyncio.to_thread(ensure_katago)
            contribution_dirs = [
                CONTRIBUTE_BASE_DIR,
                HOME_DATA_DIR,
                LOG_DIR,
            ]
            for directory in contribution_dirs:
                directory.mkdir(parents=True, exist_ok=True)

            config_text = build_config(request["config"])
            descriptor, temporary_name = tempfile.mkstemp(
                prefix="swhub_contribute_",
                suffix=".cfg",
                dir="/tmp",
                text=True,
            )
            config_path = Path(temporary_name)
            with os.fdopen(descriptor, "w", encoding="utf-8") as output:
                output.write(config_text)
            config_path.chmod(0o600)

            arguments = [
                str(katago),
                "contribute",
                "-config",
                str(config_path),
                "-base-dir",
                str(CONTRIBUTE_BASE_DIR),
            ]
            process = await asyncio.create_subprocess_exec(
                *arguments,
                stdin=asyncio.subprocess.PIPE,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
                limit=50 * 1024 * 1024,
            )
            await send(
                "started",
                message=(
                    f"KataGo {KATAGO_RELEASE} is contributing on Modal {MODAL_GPU}."
                ),
            )
            reader_tasks = [
                asyncio.create_task(read_stdout()),
                asyncio.create_task(read_stderr()),
                asyncio.create_task(periodic_commit()),
            ]

            process_wait = asyncio.create_task(process.wait())
            while process.returncode is None:
                receive = asyncio.create_task(websocket.receive_text())
                done, _ = await asyncio.wait(
                    {receive, process_wait},
                    return_when=asyncio.FIRST_COMPLETED,
                )
                if process_wait in done:
                    receive.cancel()
                    await asyncio.gather(receive, return_exceptions=True)
                    break
                raw = receive.result()
                try:
                    command_message = json.loads(raw)
                except json.JSONDecodeError:
                    await send("warning", message="Command must be valid JSON.")
                    continue
                if not isinstance(command_message, dict):
                    await send(
                        "warning",
                        message="Command must be a JSON object.",
                    )
                    continue
                command = command_message.get("action")
                if command == "pause":
                    await write_command("pause")
                    paused = True
                    await send("paused", paused=True, message="Contribution paused.")
                elif command == "resume":
                    await write_command("resume")
                    paused = False
                    await send(
                        "resumed", paused=False, message="Contribution resumed."
                    )
                elif command == "quit":
                    shutting_down = True
                    if paused:
                        await write_command("resume")
                        paused = False
                    await write_command("quit")
                    await send(
                        "stopping",
                        message=(
                            "KataGo is finishing active games and will not start "
                            "new ones."
                        ),
                    )
                elif command == "forcequit":
                    await send("stopping", message="Force-stopping KataGo…")
                    await stop_process(force=True)
                    break
                else:
                    await send(
                        "warning",
                        message=f"Unsupported contribution command: {command!r}",
                    )

            return_code = await process_wait
            await drain_output_readers()
            await send(
                "stopped",
                uploadedGames=uploaded_games,
                message=f"KataGo exited with code {return_code}.",
            )
        except WebSocketDisconnect:
            print("SWHub viewer disconnected; stopping KataGo.", flush=True)
        except asyncio.TimeoutError:
            try:
                await send("error", message="Timed out waiting for startup data.")
            except Exception:
                pass
        except Exception as exc:
            try:
                await send("error", message=f"Contributor failed: {exc}")
            except Exception:
                pass
            print(f"Contributor failed: {exc!r}", flush=True)
        finally:
            if process is not None and process.returncode is None:
                await stop_process(force=True if not shutting_down else False)
            for task in reader_tasks:
                task.cancel()
            if reader_tasks:
                await asyncio.gather(*reader_tasks, return_exceptions=True)
            if config_path is not None:
                config_path.unlink(missing_ok=True)
            if process is not None:
                await commit_cache()

    return web_app


## 4. Optional contributor endpoint login

The private Modal URL is sufficient by default. To additionally require an application-level username/password from SWHub, change `ENABLE_CONTRIBUTOR_LOGIN` to `True` and run this cell before every deploy. The password is entered with `getpass`, passed to Modal as an ephemeral `Secret.from_dict`, and is not written into the generated script or notebook. This login is separate from both the katagotraining.org account and HTTPS-proxy authentication.


In [ ]:
from getpass import getpass

ENABLE_CONTRIBUTOR_LOGIN = False
CONTRIBUTOR_LOGIN_USERNAME = ""
CONTRIBUTOR_LOGIN_PASSWORD = ""

if ENABLE_CONTRIBUTOR_LOGIN:
    CONTRIBUTOR_LOGIN_USERNAME = input("Contributor endpoint username: ").strip()
    CONTRIBUTOR_LOGIN_PASSWORD = getpass("Contributor endpoint password: ")
    if not CONTRIBUTOR_LOGIN_USERNAME or not CONTRIBUTOR_LOGIN_PASSWORD:
        raise ValueError("Both contributor endpoint login fields are required.")
    print("Optional contributor endpoint login will be enabled on deploy.")
else:
    print("Contributor endpoint login disabled; the private URL is the credential.")


## 5. Deploy

Deployment creates the endpoint but does not deliberately cold-start the paid GPU. A WebSocket connection can cold-start the T4 before application-level login is checked; successful login is required before KataGo starts. The deployment is limited to one container and one active contribution session. Keep the URL private even when optional endpoint login is enabled.


In [ ]:
import os, re, subprocess
from pathlib import Path

env = os.environ.copy()
env["NO_COLOR"] = "1"
env["TERM"] = "dumb"
if globals().get("ENABLE_CONTRIBUTOR_LOGIN", False):
    username = globals().get("CONTRIBUTOR_LOGIN_USERNAME", "").strip()
    password = globals().get("CONTRIBUTOR_LOGIN_PASSWORD", "")
    if not username or not password:
        raise ValueError("Run the optional endpoint-login cell again.")
    env["SWHUB_CONTRIBUTOR_USERNAME"] = username
    env["SWHUB_CONTRIBUTOR_PASSWORD"] = password

script_path = Path.cwd() / "modal_katago_contribute.py"
if not script_path.exists():
    raise RuntimeError("Run the Write the Modal app cell before deploying.")

deploy = subprocess.Popen(
    ["modal", "deploy", str(script_path)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
deploy_lines = []
for line in deploy.stdout:
    print(line, end="")
    deploy_lines.append(line)
if deploy.wait() != 0:
    raise RuntimeError("Modal deployment failed")

urls = []
for line in deploy_lines:
    for url in re.findall(r"https://[^\s]+?\.modal\.run", line):
        clean = url.rstrip(".,)];'")
        if clean not in urls:
            urls.append(clean)

if urls:
    base_url = urls[-1].rstrip("/")
    websocket_url = base_url.replace("https://", "wss://") + "/contribute"
    print("\nPaste this into SWHub's Contributor WebSocket endpoint field:")
    print(websocket_url)
else:
    print("\nCopy the https://...modal.run URL above, change https to wss, and add /contribute.")


## 6. Start from SWHub

1. Open SWHub.
2. Open **Home > KataGo Contribution**.
3. Enter your katagotraining.org username and unique password once; SWHub saves both in local shared preferences.
4. Paste the `wss://.../contribute` URL printed above.
5. If optional endpoint login was enabled for the deploy, enter the same endpoint username/password in the Contributor connection section.
6. Keep the default 8 simultaneous games for a T4 initially, then press **Start contributing**.

On later runs the saved account and endpoint are reused, so **Start** is enough. KataGo selects the current training model and tasks from the server; this page does not use the small/medium/large analysis-model presets.


## 7. Logs and stopping

The SWHub page shows KataGo logs, active games, visits/sec, moves, and uploads. For raw Modal logs, run the next cell. Use **Finish games & stop** to let in-progress games upload; **Force stop** abandons them. Closing the page or losing the WebSocket also force-stops the contributor for cost safety.


In [ ]:
# Uncomment to stream raw service logs.
# !modal app logs katago-contribute

# Uncomment after contribution has stopped to terminate warm containers.
# !modal app stop katago-contribute
